# DeepCompress OCR Benchmark

This notebook tests the main DeepCompress path on Google Colab:

`synthetic PDF/image dataset -> DeepSeek-OCR -> D-TOON compression -> token/fact metrics`

Run the cells top to bottom. Use a GPU runtime.

In [ ]:
# Colab setup. This installs Poppler for PDF conversion and DeepCompress OCR deps.
import os

!apt-get update -qq
!apt-get install -y -qq poppler-utils

if os.path.exists('/content/EDC/pyproject.toml'):
    %pip install -q -e "/content/EDC[colab,llm]" "transformers==4.46.3" "tokenizers==0.20.3" sentencepiece addict accelerate pdf2image tqdm pillow matplotlib pandas packaging
elif os.path.exists('pyproject.toml'):
    %pip install -q -e ".[colab,llm]" "transformers==4.46.3" "tokenizers==0.20.3" sentencepiece addict accelerate pdf2image tqdm pillow matplotlib pandas packaging
else:
    %pip install -q --no-deps deepcompress
    %pip install -q pydantic pydantic-settings aiofiles httpx redis prometheus-client python-dotenv tenacity cryptography orjson tiktoken openai anthropic "transformers==4.46.3" "tokenizers==0.20.3" sentencepiece addict accelerate pdf2image tqdm pillow matplotlib pandas packaging

# Patch installed DeepCompress in-place when running this notebook standalone from PyPI.
# This avoids DeepSeek-OCR fast-tokenizer failures such as:
#   data did not match any variant of untagged enum ModelWrapper
import deepcompress, pathlib, re, shutil
extractor_path = pathlib.Path(deepcompress.__file__).parent / 'core' / 'extractor.py'
text = extractor_path.read_text()
if 'use_fast=False' not in text and 'use_fast=self.config.ocr_use_fast_tokenizer' not in text:
    text = text.replace('trust_remote_code=True,\n            )', 'trust_remote_code=True,\n                use_fast=False,\n            )')
    text = text.replace('trust_remote_code=True,\n                    )', 'trust_remote_code=True,\n                        use_fast=False,\n                    )')
    extractor_path.write_text(text)
    print('Patched installed DeepCompress tokenizer loading:', extractor_path)
    import importlib
    import deepcompress.core.extractor as extractor_module
    import deepcompress.core.compressor as compressor_module
    importlib.reload(extractor_module)
    importlib.reload(compressor_module)
    deepcompress.DocumentCompressor = compressor_module.DocumentCompressor
    print('Reloaded patched DeepCompress modules')

hf_home = pathlib.Path(os.environ.get('HF_HOME', '/root/.cache/huggingface'))
for cache_path in [
    hf_home / 'hub' / 'models--deepseek-ai--DeepSeek-OCR',
    hf_home / 'modules' / 'transformers_modules' / 'deepseek-ai' / 'DeepSeek-OCR',
]:
    if cache_path.exists():
        shutil.rmtree(cache_path, ignore_errors=True)
        print('Cleared stale cache:', cache_path)

print('Setup complete')

In [ ]:
# Runtime and benchmark settings.
# Keep dtoon_mode='raw' for OCR-only compression. Use 'rag' or 'structured' only after setting an LLM key.
OCR_DEVICE = 'cuda:0'
OCR_MODE = 'small'          # small, base, large
DTOON_MODE = 'raw'          # raw, structured, rag
INPUT_KIND = 'pdf'          # pdf or images
TOKEN_COUNTER_MODEL = 'gpt-4o'
OUTPUT_DIR = '/content/deepcompress_ocr_results'
DATASET_DIR = '/content/deepcompress_ocr_dataset'
LIMIT_DOCS = 0              # 0 means all generated docs

# Optional for DTOON_MODE='rag' or 'structured'. Do not print secrets.
# os.environ['OPENAI_API_KEY'] = '...'
# os.environ['AZURE_OPENAI_API_KEY'] = '...'
# os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://<resource>.openai.azure.com/'
# os.environ['AZURE_OPENAI_DEPLOYMENT'] = '<deployment-name>'
# os.environ['AZURE_OPENAI_API_VERSION'] = '2024-12-01-preview'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

In [ ]:
# Check GPU and core packages.
import importlib.util

for module in ['torch', 'transformers', 'pdf2image', 'PIL', 'addict', 'accelerate']:
    print(f'{module}:', importlib.util.find_spec(module) is not None)

import torch
import transformers
import tokenizers
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('tokenizers:', tokenizers.__version__)
print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('device 0:', torch.cuda.get_device_name(0))

from packaging.version import Version
if not (Version('4.46.0') <= Version(transformers.__version__) < Version('4.47.0')):
    raise RuntimeError(f'Expected transformers 4.46.x, got {transformers.__version__}. Restart runtime and rerun setup cell.')
if not (Version('0.20.0') <= Version(tokenizers.__version__) < Version('0.21.0')):
    raise RuntimeError(f'Expected tokenizers 0.20.x, got {tokenizers.__version__}. Restart runtime and rerun setup cell.')

In [ ]:
# Generate a synthetic OCR dataset with known exact facts.
from dataclasses import dataclass
from pathlib import Path
import json
import textwrap

@dataclass
class SyntheticDocument:
    document_id: str
    title: str
    pages: list[str]
    expected_facts: list[str]

DATASET = [
    SyntheticDocument(
        document_id='loan_packet_2026',
        title='Residential Loan Review Packet',
        pages=[
            '''
            RESIDENTIAL LOAN REVIEW PACKET
            Application ID: LN-2026-00419
            Applicant: Jordan Vale
            Co-applicant: Mira Vale
            Property address: 418 Cedar Spring Road, Austin, TX 78704
            Requested loan amount: $486,000
            Purchase price: $610,000
            Down payment: $124,000
            Application date: 2026-03-15

            The application packet includes borrower disclosures, income verification,
            asset statements, debt review, and an underwriting decision memo. This
            synthetic document is intentionally verbose so the OCR compression path
            has repeated context to remove.
            ''',
            '''
            INCOME SUMMARY AND SUPPORTING DETAILS
            Employer: Analytical Engines LLC
            Payroll income: $17,000 per month
            Freelance consulting income: $3,200 per month
            Total verified monthly income: $20,200

            Income evidence table
            Source                         Monthly Amount      Evidence
            Payroll wages                  $17,000             Pay stubs and W-2
            Consulting contracts           $3,200              Invoices and deposits
            Total verified income          $20,200             Underwriter calculation
            ''',
            '''
            MONTHLY DEBT AND UNDERWRITING DECISION
            Automobile loan payment: $420 per month
            Credit card minimum payment: $180 per month
            Total monthly debt: $600
            Debt-to-income ratio: 2.97 percent
            Credit score: 740
            Underwriting action: APPROVE SUBJECT TO STANDARD CLOSING CONDITIONS

            Conditions: re-verify employment, confirm down payment funds, and confirm
            hazard insurance binder before final approval.
            ''',
        ],
        expected_facts=['LN-2026-00419', '$486,000', '$610,000', '$17,000', '$3,200', '$20,200', '$600', '2.97 percent', '740', 'APPROVE SUBJECT TO STANDARD CLOSING CONDITIONS'],
    ),
    SyntheticDocument(
        document_id='services_agreement_2026',
        title='Enterprise Services Agreement',
        pages=[
            '''
            ENTERPRISE SERVICES AGREEMENT
            Agreement ID: ESA-2026-7781
            Vendor: Northwind Analytics LLC
            Customer: Contoso Retail Group
            Effective date: 2026-03-15
            Initial term: 24 months
            Governing law: State of Washington

            The vendor will provide analytics dashboards, monthly business reviews,
            anomaly monitoring, and support for retail reporting workflows.
            ''',
            '''
            PAYMENT TERMS AND BILLING
            Monthly recurring service fee: $8,500
            Implementation fee: $14,000
            Invoice terms: net 30 from invoice date
            Late payment charge: 1.5% per month
            Vendor billing contact: billing@northwind.example.com
            Customer accounts payable: ap@contoso.example.com
            ''',
            '''
            TERMINATION AND REMEDIES
            Either party may terminate for material breach.
            Cure period: 30 days after written notice
            Confidentiality survival period: 12 months after termination
            Service credit cap: $25,000
            Liability cap: $102,000
            ''',
        ],
        expected_facts=['ESA-2026-7781', 'Northwind Analytics LLC', 'Contoso Retail Group', '2026-03-15', '24 months', '$8,500', '$14,000', 'net 30', '1.5%', '30 days', '12 months', '$25,000', '$102,000'],
    ),
]

def find_font(size):
    from PIL import ImageFont
    candidates = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf',
    ]
    for candidate in candidates:
        if Path(candidate).exists():
            return ImageFont.truetype(candidate, size=size)
    return ImageFont.load_default()

def render_page(text, title, page_number):
    from PIL import Image, ImageDraw
    width, height = 1700, 2200
    margin = 110
    font = find_font(34)
    title_font = find_font(44)
    footer_font = find_font(26)
    image = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(image)
    y = margin
    draw.text((margin, y), title, fill='black', font=title_font)
    y += 78
    draw.line((margin, y, width - margin, y), fill='black', width=3)
    y += 48
    for paragraph in textwrap.dedent(text).strip().splitlines():
        if not paragraph.strip():
            y += 30
            continue
        for line in textwrap.wrap(paragraph, width=78) or [paragraph]:
            draw.text((margin, y), line, fill='black', font=font)
            y += 54
            if y > height - 170:
                break
        if y > height - 170:
            break
    draw.line((margin, height - 125, width - margin, height - 125), fill='gray', width=2)
    draw.text((margin, height - 95), f'Synthetic OCR benchmark page {page_number}', fill='gray', font=footer_font)
    return image

def generate_dataset(dataset_dir):
    dataset_dir = Path(dataset_dir)
    dataset_dir.mkdir(parents=True, exist_ok=True)
    manifest = []
    for doc in DATASET:
        doc_dir = dataset_dir / doc.document_id
        doc_dir.mkdir(parents=True, exist_ok=True)
        images = []
        image_paths = []
        for index, page_text in enumerate(doc.pages, start=1):
            image = render_page(page_text, doc.title, index)
            image_path = doc_dir / f'page_{index:02d}.png'
            image.save(image_path)
            images.append(image)
            image_paths.append(str(image_path))
        pdf_path = doc_dir / f'{doc.document_id}.pdf'
        images[0].save(pdf_path, 'PDF', resolution=200.0, save_all=True, append_images=images[1:])
        manifest.append({'document_id': doc.document_id, 'title': doc.title, 'pdf_path': str(pdf_path), 'image_paths': image_paths, 'page_count': len(doc.pages), 'expected_facts': doc.expected_facts})
    (dataset_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    return manifest

manifest = generate_dataset(DATASET_DIR)
print('Generated documents:', len(manifest))
print('First PDF:', manifest[0]['pdf_path'])

In [ ]:
# Benchmark helpers.
import asyncio
import time
from pathlib import Path
import pandas as pd

from deepcompress import DeepCompressConfig, DocumentCompressor

def normalize_text(value):
    return ' '.join(str(value).lower().split())

def fact_recall(text, expected_facts):
    normalized = normalize_text(text)
    found = []
    missing = []
    for fact in expected_facts:
        if normalize_text(fact) in normalized:
            found.append(fact)
        else:
            missing.append(fact)
    score = len(found) / len(expected_facts) if expected_facts else 1.0
    return len(found), len(expected_facts), score, missing

def extracted_text(result):
    return '\n\n'.join(page.raw_text.strip() for page in result.extracted.pages if page.raw_text and page.raw_text.strip())

def preview(text, limit=500):
    text = ' '.join(str(text).split())
    return text if len(text) <= limit else text[:limit] + '...'

def build_config():
    llm_key = os.getenv('OPENAI_API_KEY') or os.getenv('AZURE_OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY') or ''
    return DeepCompressConfig(
        dtoon_mode=DTOON_MODE,
        protect_facts=True,
        ocr_device=OCR_DEVICE,
        ocr_mode=OCR_MODE,
        ocr_batch_size=1,
        vector_db_provider='none',
        cache_enabled=False,
        llm_provider='openai',
        llm_api_key=llm_key,
        token_counter_provider='openai',
        token_counter_model=TOKEN_COUNTER_MODEL,
    )

async def run_benchmark():
    docs = manifest[:LIMIT_DOCS] if LIMIT_DOCS else manifest
    config = build_config()
    compressor = DocumentCompressor(config)
    rows = []
    started = time.perf_counter()
    for item in docs:
        targets = [(item['document_id'], item['pdf_path'], item['expected_facts'])]
        if INPUT_KIND == 'images':
            targets = [(f"{item['document_id']}_page_{i}", path, item['expected_facts']) for i, path in enumerate(item['image_paths'], start=1)]
        for document_id, source_path, expected_facts in targets:
            print('Compressing:', document_id, source_path)
            result = await compressor.compress(source_path, document_id=document_id)
            ocr_text = extracted_text(result)
            ocr_found, ocr_total, ocr_score, ocr_missing = fact_recall(ocr_text, expected_facts)
            cmp_found, cmp_total, cmp_score, cmp_missing = fact_recall(result.optimized_text, expected_facts)
            rows.append({
                'document_id': document_id,
                'source_path': source_path,
                'page_count': result.extracted.page_count,
                'original_tokens': result.original_tokens_measured,
                'compressed_tokens': result.compressed_tokens_measured,
                'tokens_saved': result.tokens_saved,
                'compression_ratio': result.compression_ratio_measured,
                'processing_time_s': result.processing_time_ms / 1000,
                'ocr_fact_recall': ocr_score,
                'ocr_facts': f'{ocr_found}/{ocr_total}',
                'ocr_missing_facts': ocr_missing,
                'compressed_fact_recall': cmp_score,
                'compressed_facts': f'{cmp_found}/{cmp_total}',
                'compressed_missing_facts': cmp_missing,
                'ocr_text_preview': preview(ocr_text),
                'optimized_text_preview': preview(result.optimized_text),
            })
    elapsed = time.perf_counter() - started
    return rows, elapsed

rows, elapsed_s = await run_benchmark()
df = pd.DataFrame(rows)
df

In [ ]:
# Summary table.
summary = {
    'documents': len(df),
    'original_tokens': int(df['original_tokens'].sum()),
    'compressed_tokens': int(df['compressed_tokens'].sum()),
    'tokens_saved': int(df['tokens_saved'].sum()),
    'compression_ratio': float(df['original_tokens'].sum() / df['compressed_tokens'].sum()) if df['compressed_tokens'].sum() else 1.0,
    'token_reduction': float(1 - (df['compressed_tokens'].sum() / df['original_tokens'].sum())) if df['original_tokens'].sum() else 0.0,
    'ocr_fact_recall': float(df['ocr_fact_recall'].mean()),
    'compressed_fact_recall': float(df['compressed_fact_recall'].mean()),
    'elapsed_s': elapsed_s,
}
summary_df = pd.DataFrame([summary])
summary_df

In [ ]:
# Charts.
import matplotlib.pyplot as plt

chart_df = df.copy()
labels = chart_df['document_id'].tolist()

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(chart_df))
width = 0.38
ax.bar([i - width / 2 for i in x], chart_df['original_tokens'], width, label='Full OCR text')
ax.bar([i + width / 2 for i in x], chart_df['compressed_tokens'], width, label='Compressed output')
ax.set_title('OCR Text Tokens vs Compressed Tokens')
ax.set_ylabel('Tokens')
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=20, ha='right')
ax.legend()
plt.tight_layout()
tokens_chart = os.path.join(OUTPUT_DIR, 'ocr_tokens.png')
plt.savefig(tokens_chart, dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(labels, chart_df['compression_ratio'])
ax.axhline(1.0, color='black', linewidth=1)
ax.set_title('Compression Ratio by Document')
ax.set_ylabel('Original tokens / compressed tokens')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
ratio_chart = os.path.join(OUTPUT_DIR, 'ocr_compression_ratio.png')
plt.savefig(ratio_chart, dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i - width / 2 for i in x], chart_df['ocr_fact_recall'] * 100, width, label='OCR text')
ax.bar([i + width / 2 for i in x], chart_df['compressed_fact_recall'] * 100, width, label='Compressed output')
ax.set_title('Exact Fact Recall')
ax.set_ylabel('Recall (%)')
ax.set_ylim(0, 105)
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=20, ha='right')
ax.legend()
plt.tight_layout()
fact_chart = os.path.join(OUTPUT_DIR, 'ocr_fact_recall.png')
plt.savefig(fact_chart, dpi=160)
plt.show()

print(tokens_chart)
print(ratio_chart)
print(fact_chart)

In [ ]:
# Export JSON, CSV, and Markdown report.
import json

results = {
    'benchmark': 'ocr_colab_notebook',
    'input_kind': INPUT_KIND,
    'ocr_device': OCR_DEVICE,
    'ocr_mode': OCR_MODE,
    'dtoon_mode': DTOON_MODE,
    'summary': summary,
    'documents': df.to_dict(orient='records'),
}

json_path = os.path.join(OUTPUT_DIR, 'ocr_colab_results.json')
csv_path = os.path.join(OUTPUT_DIR, 'ocr_colab_results.csv')
md_path = os.path.join(OUTPUT_DIR, 'ocr_colab_results.md')

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
df.to_csv(csv_path, index=False)

md_lines = [
    '# DeepCompress OCR Colab Results',
    '',
    f'- Input kind: `{INPUT_KIND}`',
    f'- OCR device: `{OCR_DEVICE}`',
    f'- OCR mode: `{OCR_MODE}`',
    f'- D-TOON mode: `{DTOON_MODE}`',
    '',
    '## Summary',
    '',
    summary_df.to_markdown(index=False),
    '',
    '## Per Document',
    '',
    df[['document_id','page_count','original_tokens','compressed_tokens','tokens_saved','compression_ratio','ocr_fact_recall','compressed_fact_recall','processing_time_s']].to_markdown(index=False),
    '',
    '## Charts',
    '',
    f'- {tokens_chart}',
    f'- {ratio_chart}',
    f'- {fact_chart}',
]
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(md_lines))

print('Wrote:')
print(json_path)
print(csv_path)
print(md_path)
print(OUTPUT_DIR)

In [ ]:
# Optional: zip the outputs for download from Colab.
import shutil
zip_path = shutil.make_archive('/content/deepcompress_ocr_results', 'zip', OUTPUT_DIR)
print(zip_path)